# Distributed Energy Resources

----

## Inverter Power Flow Modeling 

CIM 17 has introduced detailed modeling of distributed energy resources (DERs) with even more detailed models to be added in the upcoming draft CIM 18. The first set of modeling classes are contained within the Wires and Production packages. The inverter is specified as a PowerElectronicsConnection with attributes for the rated voltage and maximum real / reactive / apparent power that can be produced by the inverter. Each PowerElectronicsConnection inverter is associated with a single Terminal object on the AC side of the device. No explicit modeling of the DC connectivity is included. Single-phase inverters can be specified by defining each phase component as a PowerElectronicsConnectionPhase associated with the overall PowerElectronicsConnection. The DC source behind the inverter is specified through Production package as a PhotoVoltaicUnit, BatteryUnit, or PowerElectronicsWindUnit. The minimum and maximum power of each DC source is specified through the source class, as shown in Figure 28. Basic inverter control modes are specified as an enumeration.

In [1]:
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cim17v40 as cim

In [ ]:
diagram_text = utils.get_mermaid([cim.RegulatingCondEq, cim.RegulatingControl, cim.Terminal, cim.PowerElectronicsConnection, cim.PowerElectronicsUnit, cim.PowerElectronicsWindUnit, cim.BatteryUnit, cim.PhotoVoltaicUnit, cim.PowerElectronicsConnectionPhase, cim.SinglePhaseKind, cim.BatteryStateKind]) ## 
Mermaid(diagram_text)

## IEEE 1547-2018 Inverter Dynamics Modeling

IEC Standard 61970-302, 2nd Edition has added an initial set of inverter 61970 dynamics classes, which have been extended further by the GridAPPS-D project in the CIMHub package [4], [5]. The approach taken in this initial version is similar to that of transmission generator dynamics modeling with a set of dynamics objects defined for each inverter. The upcoming draft CIM 18 will likely revise these definitions to introduce an asset-based approach with 61968 standard nameplate definitions for common DERs which can be defined once and then referenced across the entire service territory of a utility.

The DERIEEEType1 class is used to describe smart inverter functions and other DER behavior during time-series power flow, as shown in Figure 30. The CIM classes and attributes generally map to the interoperability tables in the IEEE standards, except for different capitalization convention, with the use of camelCase by CIM and the use of capital letters and underscores in the IEEE tables. Both PowerElectronicsConnection inverters and RotatingMachine objects can be associated to DERIEEEType1 for supplemental nameplate and rating information in the network model. Preliminary values for these attributes would be available from an application to interconnect DER, and then updated as the project moves through commissioning to operational status. Detailed descriptions of the classes and attributes are available from IEEE 1547-2018 [6], IEEE 1547.1-2020 [7], and IEEE P1547.2/D6.2 (Annex F).

In [16]:
# diagram_text = utils.get_mermaid([cim.PowerLimitSetting]) 
# Mermaid(diagram_text)

Some examples are discussed below.

In [3]:
from cimgraph.models import FeederModel
from cimgraph.databases import ConnectionParameters, XMLFile
import cimgraph.data_profile.cimhub_2023 as cim
import json
from uuid import UUID

In [4]:
params = ConnectionParameters(filename='../sample_models/ieee13.xml',
                              cim_profile='cimhub_2023',
                              iec61970_301=8) # file path
file = XMLFile(params) # file read connection
network = FeederModel(container=cim.Feeder(),connection=file) # create feeder model

Example 1: What is the state of charge the inverter with mRID "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"?

In [5]:
result = None
mRID = "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"

# Final attribute is float(cim.BatteryUnit.storedE)/float(cim.BatteryUnit.ratedE)

# Convert the mRID to a UUID object
uuid = UUID(mRID.strip('_').lower())

# Get the inverter with correct uuid, which is a PowerElectronicsConnection in the graph
inverter = network.graph[cim.PowerElectronicsConnection][uuid]
# Iterate through all PowerElectronicsUnit connected to the inverter
for unit in inverter.PowerElectronicsUnit:
    # Check if it is a BatteryUnit
    if isinstance(unit, cim.BatteryUnit):
        # Get the total amount of energy stored in the battery
        stored = unit.storedE
        # Get the total capacity of the battery
        capacity = unit.ratedE
        # State of charge is the stored energy divided by the capacity
        state_of_charge = float(stored) / float(capacity)
        result = state_of_charge

print(result)

0.37037037037037035


Example 2:  What is the charging status of all batteries in the model?

In [6]:
result = []

# Iterate through all BatteryUnit objects in the graph
for battery in network.graph[cim.BatteryUnit].values():
    # Charging status is given by the BatteryUnit.batteryState attributes
    # The value is an enumeration of BatteryStateKind
    result.append(str(battery.batteryState))
    
print(result)

['BatteryStateKind.discharging', 'BatteryStateKind.waiting', 'BatteryStateKind.charging']


Example 3: What is the nominal rated voltage of the inverter named "house"?

In [7]:
result = []
name = "house"

# Final attribute is cim.PowerEletronicsConnection.ratedU
# This class does not use BaseVoltage

# Loop through all inverters given by the PowerElectronicsConnection class
for inverter in network.graph[cim.PowerElectronicsConnection].values():
    # Check if the name matches
    if name in inverter.name:
        # Get the rated nominal voltage of the inverter
        result = inverter.ratedU

print(result)

208.0


Example 4: What are the rated capacities of the batteries connected to the bus node named "634"?

In [8]:
result = []
name = "634"

# Starting attribute is cim.ConnectivityNode.name
# Final attribute is cim.BatteryUnit.ratedE
# Graph traversal path is ConnectivityNode -> List[Terminal] -> ConductingEquipment -> List[PowerElectronicsUnit]

# Iterate through all ConnectivityNode objects in the network graph
if cim.ConnectivityNode in network.graph:
    for node in network.graph[cim.ConnectivityNode].values():
        # Check if the current node's name contains the specified bus node name
        if name in node.name:
            # Iterate through all terminals associated with the current node
            for terminal in node.Terminals:
                # Get the conducting equipment connected to the terminal
                equipment = terminal.ConductingEquipment
                # Check if the equipment is a PowerElectronicsConnection
                if isinstance(equipment, cim.PowerElectronicsConnection):
                    # Iterate through all PowerElectronicsUnits associated with the equipment
                    for unit in equipment.PowerElectronicsUnit:
                        # Check if the unit is a type of BatteryUnit
                        if isinstance(unit, cim.BatteryUnit):
                            # Append the rated capacity (ratedE) of the battery to the result list
                            value = dict()
                            value['rated_capacity'] = unit.ratedE
                            result.append(value)

# Print the result list containing the rated capacities of the batteries connected to the specified node
print(result)

[{'rated_capacity': 200000.0}, {'rated_capacity': 200000.0}]


Example 5: What bus is the battery named "school" connected to?

In [9]:
result = []
name = "school"

# Final attribute is cim.ConnectivityNode.name
# Graph traveral path is BatteryUnit -> PowerElectronicsConnection -> Terminal -> ConnectivityNode

# Iterate through all BatteryUnit objects in the network graph
for battery in network.graph[cim.BatteryUnit].values():
    # Check if the current battery's name matches the specified name
    if name in battery.name:
        # Retrieve the associated PowerElectronicsConnection (Inverter) for the battery
        inverter = battery.PowerElectronicsConnection
        # Iterate through all terminals of the PowerElectronicsConnection
        for terminal in inverter.Terminals:
            # Get the ConnectivityNode (bus) connected to the terminal
            bus = terminal.ConnectivityNode
            # Append the name of the bus to the result list
            result.append(bus.name)

# Print the result list containing the names of the buses the battery is connected to
print(result)

['634']


Example 6: What are the names of the batteries connected to the bus node named "634"?

In [10]:
result = []
name = "634"
        
# Final attribute is cim.BatteryUnit.name
# Graph traversal path is ConnectivityNode -> List[Terminal] -> ConductingEquipment -> List[PowerElectronicsUnit]

# Iterate through all ConnectivityNode objects in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the current node's name contains the specified bus node name
    if name in node.name:
        # Iterate through all terminals associated with the current node
        for terminal in node.Terminals:
            # Get the conducting equipment connected to the terminal
            equipment = terminal.ConductingEquipment
            # Check if the equipment is an inverter with class PowerElectronicsConnection
            if isinstance(equipment, cim.PowerElectronicsConnection):
                # Iterate through all PowerElectronicsUnit associated with the equipment
                for unit in equipment.PowerElectronicsUnit:
                    # Check if the unit is a type of BatteryUnit
                    if isinstance(unit, cim.BatteryUnit):
                        # Append the name of the battery unit to the result list
                        result.append(unit.name)
print(result)

['school', 'batidle']


Example 7: What is the rated output of the inverter connected to the bus node named "634"?

In [11]:
result = []
name = "634"
        
# Final attribute is PowerElectronicsConnection.ratedS, which is apparent power in VA or kVA
# Graph traversal path is ConnectivityNode -> List[Terminal] -> ConductingEquipment

# Iterate through all ConnectivityNode objects in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the current node's name contains the specified bus node name
    if name in node.name:
        # Iterate through all terminals associated with the current node
        for terminal in node.Terminals:
            # Get the conducting equipment connected to the terminal
            equipment = terminal.ConductingEquipment
            # Check if the equipment is a PowerElectronicsConnection (Inverter)
            if isinstance(equipment, cim.PowerElectronicsConnection):
                # Append the rated output (ratedS) of the inverter to the result list
                result.append(equipment.ratedS)

# Print the result list containing the rated output of the inverters connected to the specified bus node
print(result)

[300000.0, 100000.0, 100000.0]


Example 8: What is the rated voltage of the inverter connected to battery named "house"?


In [12]:
result = []
name = "house"

# Final attribute is PowerElectronicsConnection.ratedU
# Inverters do not have an association to BaseVoltage

# Iterate through all BatteryUnit objects in the graph
for battery in network.graph[cim.BatteryUnit].values():
    # Check if the name matches
    if name in battery.name:
        # Get the inverter to which the battery is connected
        inverter = battery.PowerElectronicsConnection
        # nominal voltage of an inverter is ratedU
        result.append(inverter.ratedU)
         
print(result)

[208.0]


Example 9: What is the rated capacity of the battery for the inverter with mRID "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"?


In [13]:
result = []
mRID = "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"

# Final attribute is BatteryUnit.ratedE
# Graph traversal path is PowerElectronicsConnection -> List[PowerElectronicsUnit]

# Convert the mRID to a UUID object
uuid = UUID(mRID.strip('_').lower())

# get the inverter with the correct uuid
inverter = network.graph[cim.PowerElectronicsConnection][uuid]
# Iterate through list of all PowerElectronicsUnit to find battery connected to the inverter
for unit in inverter.PowerElectronicsUnit:
    # check if the PowerElectronicsConnection.PowerElectronicsUnit is a BatteryUnit
    if isinstance(unit, cim.BatteryUnit):
        # get the rated storage capcity which is ratedE in watt-hour or kWh or MWh
        result.append(unit.ratedE)

print(result)

[13500.0]


Example 10: What is the nominal rated kVA apparent power output of the inverter named "house"?


In [15]:
result = []
name = "house"

# Final attribute is cim.PowerElectronicsConnection.ratedS

# Iterate through all inverters in the graph model
for inverter in network.graph[cim.PowerElectronicsConnection].values():
    # Check in the name matches
    if name == inverter.name:
        # get the apparent power rating in kVA or VA
        result.append(inverter.ratedS)

print(result)

[5000.0, 5000.0]


Example 11: How much energy is stored in battery for the inverter with mRID "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"?

In [17]:
result = None
mRID = "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"

# Final attribute is cim.BatteryUnit.storedE

# Convert the mRID to a UUID object
uuid = UUID(mRID.strip('_').lower())

# Get the inverter with correct uuid, which is a PowerElectronicsConnection in the graph
inverter = network.graph[cim.PowerElectronicsConnection][uuid]
# Iterate through all PowerElectronicsUnit connected to the inverter
for unit in inverter.PowerElectronicsUnit:
    # Check if it is a BatteryUnit
    if isinstance(unit, cim.BatteryUnit):
        # Get the total amount of energy stored in the battery
        result = unit.storedE

result

5000.0

Example 12: What is the state of charge the inverter with mRID "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"?

In [18]:
result = None
mRID = "682AB7A9-4FBF-4204-BDE1-27EAB3425DA0"

# Final attribute is float(cim.BatteryUnit.storedE)/float(cim.BatteryUnit.ratedE)

# Convert the mRID to a UUID object
uuid = UUID(mRID.strip('_').lower())

# Get the inverter with correct uuid, which is a PowerElectronicsConnection in the graph
inverter = network.graph[cim.PowerElectronicsConnection][uuid]
# Iterate through all PowerElectronicsUnit connected to the inverter
for unit in inverter.PowerElectronicsUnit:
    # Check if it is a BatteryUnit
    if isinstance(unit, cim.BatteryUnit):
        # Get the total amount of energy stored in the battery
        stored = unit.storedE
        # Get the total capacity of the battery
        capacity = unit.ratedE
        # State of charge is the stored energy divided by the capacity
        state_of_charge = float(stored) / float(capacity)
        result = state_of_charge

print(result)

0.37037037037037035


Example 13: What is the largest solar panel in the model?

In [19]:
result = []

# Iterate through all inverters in the graph
for inverter in network.graph[cim.PowerElectronicsConnection].values():
    # Iterate through all PowerElectronicsConnection.PowerElectronicsUnit
    for unit in inverter.PowerElectronicsUnit:
        # Check if it is a solar panel
        if isinstance(unit, cim.PhotovoltaicUnit):
            # get the size of each solar panel
            result.append(float(unit.maxP))

# Get the largest value
result = max(result)

result 

300000.0

Example 14: What is the reactive power rating of the inverter named "house"?

In [20]:
result = None
name = "house"

# iterate through all PowerElectronicsConnection to find the inverter
for inverter in network.graph[cim.PowerElectronicsConnection].values():
    # check if the name matches
    if name == inverter.name:
        value = dict()
        # the maximum amount of reactive power the inverter can absorb in minQ
        value['minQ'] = inverter.minQ
        # the maxmimium amount reactive power the inverter can inject is maxQ
        value['maxQ'] = inverter.maxQ
        result = value

print(result)

{'minQ': -5000.0, 'maxQ': 5000.0}


Example 15: What is the real power rating of the solar panel for inverter named "house"?

In [21]:
result = None
name = "house"

# Final attribute is cim.PhotovoltaticUnit.maxP
# Graph traversal path is PowerElectronicsConnection -> List[PowerElectronicsUnit]

# Iterate through all PowerElectronicsConnection
for inverter in network.graph[cim.PowerElectronicsConnection].values():
    # check if the name matches
    if name == inverter.name:
        # Loop through all PowerElectronicsUnit connected to the inverter to find a solar panel
        for unit in inverter.PowerElectronicsUnit:
            # check if it is a solar panel, represented by PhotovoltaticUnit class
            if isinstance(unit, cim.PhotovoltaicUnit):
                # The solar panel real power rating is given by maxP
                result = unit.maxP

print(result)

5000.0


Example 16: Create a new solar panel sized at 5 kW with an inverter with make Enphase6000 sized at 6 kVA connected to bus named 634?

In [22]:
# Create a new inverter
new_inverter = cim.PowerElectronicsConnection()
# Generate a new mRID for the inverter
new_inverter.uuid()
# Add the inverter to the graph
network.add_to_graph(new_inverter)
# Assign the size as the kVA rating which is PowerElectronicsConnection.ratedS
new_inverter.ratedS = 6000
# Add the make/model as the name
new_inverter.name = 'Enphase6000'

# Create a new solar panel
new_solar = cim.PhotovoltaicUnit()
# Generate a new mRID for the solar panel:
new_solar.uuid()
# Add the solar panel to the graph
network.add_to_graph(new_solar)
# Assign the size as the kW rating which is PowerElectronicsUnit.maxP
new_solar.maxP = 5000

# Associate the inverter to the solar panel
new_solar.PowerElectronicsConnection = new_inverter
# Add the solar panel to inverter's list of PowerElectronicsUnit
new_inverter.PowerElectronicsUnit.append(new_solar)

# Create a new terminal to connect the inverter to the node
new_terminal = cim.Terminal()
# Generate an mRID for the terminal
new_terminal.uuid()
# Add the terminal to the graph
network.add_to_graph(new_terminal)

# Connect the inverter to the terminal
new_terminal.ConductingEquipment = new_inverter
# Add the terminal to inverter's list of terminals
new_inverter.Terminals.append(new_terminal)
# Find the correct bus named 634
for node in network.graph[cim.ConnectivityNode].values():
    if node.name == '634':
        # connect the terminal to the matching node
        new_terminal.ConnectivityNode = node
        # add the terminal to the node's list of terminals
        node.Terminals.append(new_terminal)

print(new_inverter)
print(new_solar)

{"@id": "d9b7dbe2-2aed-49e6-9262-1a2b20e41984", "@type": "PowerElectronicsConnection", "name": "Enphase6000", "Terminals": [{"@id": "f4cf7158-3c30-4e2a-a25e-4530944004fe", "@type": "Terminal"}], "ratedS": "6000", "PowerElectronicsUnit": [{"@id": "853a767c-e410-405e-a9dc-61ba7cf85fd1", "@type": "PhotovoltaicUnit"}]}
{"@id": "853a767c-e410-405e-a9dc-61ba7cf85fd1", "@type": "PhotovoltaicUnit", "maxP": "5000", "PowerElectronicsConnection": {"@id": "d9b7dbe2-2aed-49e6-9262-1a2b20e41984", "@type": "PowerElectronicsConnection"}}
